# Ungraded Lab -  Retrieval Metrics
# 未分级的实验室-检索指标
---

In this lab, you will be working on retrieving and analyzing metrics for a RAG system. RAG models are designed to improve the quality of generated responses by retrieving relevant documents from a knowledge base. Your goal is to evaluate the retrieval component by calculating precision and recall metrics, along with context precision and context recall.

在本实验中，您将致力于检索和分析RAG系统的度量。RAG模型旨在通过从知识库检索相关文档来提高生成响应的质量。您的目标是通过计算精度和召回度量，以及上下文精度和上下文召回来评估检索组件。

In this lab, you will learn:
- How to compute precision and recall metrics
- How to apply these metrics in information retrieval
- How to work with a concrete dataset to test the retrieval capabilities of semantic-based searches

在本实验中，您将学习：
- 如何计算精度和召回指标
- 如何将这些指标应用于信息检索
- 如何使用具体的数据集来测试基于语义的搜索的检索能力

You will be using the `sentence-transformers` library to convert text to embeddings, allowing efficient similarity computations. To compute retrieval metrics, you need a labeled dataset.

您将使用`句子转换器`库将文本转换为嵌入，从而实现高效的相似度计算。为了计算检索指标，您需要一个标记的数据集。

---
<h4 style="color:black; font-weight:bold;">USING THE TABLE OF CONTENTS</h4>

<h4 style="color:black; font-weight:bold;">使用目录表</h4>

JupyterLab provides an easy way for you to navigate through your assignment. It's located under the Table of Contents tab, found in the left panel, as shown in the picture below.

JupyterLab为您提供了一种简单的方法来浏览您的作业。它位于左侧面板的目录选项卡下，如下图所示。

![TOC Location](images/toc.png)


# Table of Contents
- [ 1 - The dataset](#1)
  - [ 1.1 Preprocessing and Vectorizing Data](#1-1)
  - [ 1.2 Basic functions for retrieve](#1-2)
- [ 2 - Retrieving metric](#2)
  - [ 2.1 Precision](#2-1)
  - [ 2.2 Recall](#2-2)
  - [ 2.3 Computing metrics over some queries](#2-3)


# 目录
- [1 -数据集]（#1）
- [1.1数据预处理和向量化]（#1-1）
- 1.2检索的基本功能]（#1-2）
- [2 -检索指标]（#2）
- [2.1精度]（#2-1）
- [2.2召回]（#2-2）
- [2.3计算某些查询的度量]（#2-3）

## 1 - Introduction
## 1 - 介绍
---

Retrieval metrics are fundamental in RAG systems, as they provide a way to measure performance. To effectively gauge performance, you need a labeled dataset—one where the answers to specific queries are known—allowing you to compare these results with those generated by your RAG system. In this lab, you will use a pre-labeled dataset and focus on Precision and Recall metrics.

检索指标是 RAG（检索增强生成）系统的基础，因为它们提供了一种衡量系统性能的方法。为了有效地评估性能，你需要一个带标签的数据集——即针对特定查询的正确答案是已知的数据集——从而使你能够将这些已知结果与你的 RAG 系统生成的结果进行对比。在本次实验中，你将使用一个预先标注好的数据集，并重点关注**精确率（Precision）和召回率（Recall）**这两个指标。

<div style="text-align: center;">
  <img src="images/precision_recall.png" alt="Description" style="width: 70%;">
</div>

In [1]:
import pandas as pd  
# 作用：数据处理的大杀器。主要用于处理表格型数据（DataFrame），
# 比如读取 CSV、Excel 文件，进行数据清洗、筛选和行列操作。

from sentence_transformers import SentenceTransformer  
# 作用：NLP 核心工具。用于加载预训练的模型，
# 将句子或段落转换为高维向量（Embeddings），让计算机能“理解”语义。

import numpy as np  
# 作用：数值计算基石。提供多维数组对象和大量的数学函数库，
# 处理那些由文本转化而来的向量矩阵时，它是效率最高、最基础的工具。

import matplotlib.pyplot as plt  
# 作用：数据可视化。用于绘制各种图表（如折线图、散点图、柱状图等），
# 在你的项目中，可能被用来展示数据分布或降维后的向量聚类结果。

import joblib  
# 作用：对象序列化与持久化。比 Python 自带的 pickle 更高效，
# 常用于将训练好的模型或处理好的大规模数据保存到硬盘，或者从硬盘加载回来。

import os  
# 作用：操作系统接口。用于处理文件路径、创建文件夹、列出目录内容等，
# 确保你的程序在不同电脑上都能正确找到文件存放的位置。

<a id='1'></a>
### 1.1 The dataset

The [20 Newsgroups dataset](https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html) is a classic text dataset with text data on various topics, with labeled categories. Let's use the `sklearn.datasets` module to load this dataset.

<a id='1'></a>
1.1 数据集

20 Newsgroups 数据集 是一个经典的文本数据集，包含关于各种主题的文本数据，并且带有已标注的类别。让我们使用 sklearn.datasets 模块来加载这个数据集。

In [2]:
from sklearn.datasets import fetch_20newsgroups
# 作用：从 scikit-learn 库中导入获取“20 Newsgroups”数据集的函数。
# 这是一个经典的文本分类数据集，包含约 20,000 篇新闻组文档。

# 1. 解析你的 Mac 桌面自定义路径
# os.path.expanduser("~") 会自动识别你的 Mac 用户名
custom_data_home = os.path.expanduser("~/Desktop/AIAgent/models/dataset")

print(f"准备获取数据集，存放路径为：{custom_data_home}")

# 加载 20 Newsgroups 数据集
newsgroups_train = fetch_20newsgroups(
    subset='train',       # 仅加载“训练”子集（通常还有 'test' 或 'all'）
    shuffle=True,         # 随机打乱数据顺序，防止样本分布过于集中
    random_state=42,      # 设置随机种子，确保你每次运行代码得到的结果是一致的（方便复现）
    data_home=custom_data_home # 指定数据集下载并保存到本地的文件夹路径
)

# 将数据集转换为 DataFrame 格式，以便更高效、直观地处理
df = pd.DataFrame({
    'text': newsgroups_train.data,      # 第一列 'text'：存储新闻的原始文本内容
    'category': newsgroups_train.target # 第二列 'category'：存储新闻所属类别的编号（如 0, 1, 2...）
})

# 打印数据集的前 5 行，预览数据结构
print(df.head())

# 打印数据集的大小（行数，列数），让你了解样本总量
print("\nDataset Size:", df.shape)

# 打印类别总数（20 Newsgroups 通常包含 20 个不同主题的类别）
print("\nNumber of Categories:", len(newsgroups_train.target_names))

# 打印所有类别的具体名称（例如：sci.med, rec.autos, talk.politics.guns 等）
print("\nCategories:", newsgroups_train.target_names)

准备获取数据集，存放路径为：/Users/a1-6/Desktop/AIAgent/models/dataset
                                                text  category
0  From: lerxst@wam.umd.edu (where's my thing)\nS...         7
1  From: guykuo@carson.u.washington.edu (Guy Kuo)...         4
2  From: twillis@ec.ecn.purdue.edu (Thomas E Will...         4
3  From: jgreen@amber (Joe Green)\nSubject: Re: W...         1
4  From: jcm@head-cfa.harvard.edu (Jonathan McDow...        14

Dataset Size: (11314, 2)

Number of Categories: 20

Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [3]:
print(f"TEXT:\n\t{df['text'][0]}\nCATEGORY:\n\t{newsgroups_train.target_names[df['category'][0]]}")
# --------------------------------------------------------------------------------
# f"..." : 这是一个 f-string（格式化字符串），允许在字符串中直接嵌入变量或表达式。
# \n     : 换行符，让输出跳转到下一行。
# \t     : 制表符（Tab），让输出缩进，起到对齐美化的作用。
#
# {df['text'][0]} :
# 访问 DataFrame 中 'text' 列的第 0 行（即第一条新闻的原始内容）。
#
# {newsgroups_train.target_names[df['category'][0]]} :
# 这里包含两步逻辑：
# 1. df['category'][0]：获取第一条样本的类别编号（例如得到数字 7）。
# 2. newsgroups_train.target_names[...]：用这个编号作为索引，去类别名称列表中查找
#    对应的文字标签（例如将数字 7 转换成人类可读的 "rec.autos"）。
# --------------------------------------------------------------------------------

TEXT:
	From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Thanks,
- IL
   ---- brought to you by your neighborhood Lerxst ----





CATEGORY:
	rec.autos


<a id='1-1'></a>
### 1.1 Preprocessing and Vectorizing Data

In this section, you'll preprocess the text data by cleaning it and then vectorize the text using a pre-trained model from the `sentence-transformers` library. You will use the model `BAAI/bge-base-en-v1.5` for encoding the sentences into vectors. To save time, the dataset has been embedded ahead of time for you, so the model will be used only to vectorize the prompts.

### 1.1数据预处理和向量化
在本节中，您将通过清理文本数据来预处理文本数据，然后使用来自`句子转换器`库的预训练模型对文本进行矢量化。您将使用`BAAI/bge-base-en-v1.5`模型将句子编码为向量。为了节省时间，数据集已经提前为您嵌入，因此该模型将仅用于向量化提示。

In [5]:
# 1. 定义你的自定义存放路径（自动解析 Mac 用户主目录）
custom_path = os.path.expanduser("~/Desktop/AIAgent/models")
# 指定要使用的预训练模型名称
model_name = "BAAI/bge-base-en-v1.5"
# 作用：定义模型 ID。BAAI（北京人工智能研究院）开发的 BGE 模型是目前 NLP 领域性能极强的
# 向量化模型，v1.5 是其中的一个优化版本，专门用于提高语义检索的准确度。

# 加载句子转换器（Sentence Transformer）模型
model = SentenceTransformer(model_name, cache_folder=custom_path)
print(f"模型加载成功，路径指向：{custom_path}")
# 作用：这是真正的“加载模型”动作。
# 1. os.environ['MODELS']：从系统环境变量中获取存储模型的根目录路径。
# 2. os.path.join(...)：将根目录和模型名称拼接成完整的文件夹路径。
# 3. SentenceTransformer(...)：将该路径下的深度学习权重文件加载到内存（或 GPU），
#    准备好将文本转化为数字向量。

# 从硬盘加载预先计算好的向量数据
embedding_vectors = joblib.load('embeddings.joblib')
# 作用：读取之前保存过的向量文件。
# 将成千上万条文本转化成向量是非常耗时的，通过 joblib.load，你可以直接把上次算好的
# 结果读回来，实现“秒开”的效果，而不需要重新跑一遍模型推理。

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


模型加载成功，路径指向：/Users/a1-6/Desktop/AIAgent/models


In [7]:
len(embedding_vectors)

11314

<a id='1-2'></a>
### 1.2 Basic functions for retrieval

Now let's implement a basic RAG mechanism by performing a similarity search over our precomputed embeddings. This code uses cosine similarity to find the most relevant documents for a given query. Let's first define our basic functions.

### 1.2检索的基本功能
现在让我们通过对预先计算的嵌入执行相似性搜索来实现基本的RAG机制。这段代码使用余弦相似度为给定查询查找最相关的文档。让我们首先定义基本函数。


In [8]:
import numpy as np

def preprocess_text(text):
    """
    通过移除首尾空格来预处理文本数据。

    参数:
    text (str): 需要预处理的输入文本。

    返回:
    str: 预处理后的文本，去除了首尾空格。
    """
    # 示例预处理：调用 strip() 移除字符串开头和结尾的空格、换行符等
    text = text.strip()
    # 返回处理后的字符串
    return text


def cosine_similarity(v1, array_of_vectors):
    """
    计算一个向量与单个向量（1D）或一组向量（2D）之间的余弦相似度。
    输入为 1D 时返回浮点数，输入为 2D 时返回浮点数列表。
    能够安全地处理 PyTorch 张量（将其移动至 CPU）和 NumPy 数组。
    """
    # --- 处理向量 v1 ---
    # 检查 v1 是否具有 "detach" 属性，如果有，说明它是 PyTorch 张量
    if hasattr(v1, "detach"):  
        # 将张量从计算图中分离，移动到 CPU，并转换为 NumPy 数组
        v1 = v1.detach().cpu().numpy()
    # 确保 v1 是 float32 类型的 NumPy 数组，并展平为一维 (ravel)
    v1 = np.asarray(v1, dtype=np.float32).ravel()

    # --- 处理对比向量组 array_of_vectors ---
    # 同样检查 array_of_vectors 是否为 PyTorch 张量
    if hasattr(array_of_vectors, "detach"):  
        # 分离张量，移动到 CPU，并转换为 NumPy 数组
        array_of_vectors = array_of_vectors.detach().cpu().numpy()
    # 将输入的向量数据转换为 float32 类型的 NumPy 数组
    A = np.asarray(array_of_vectors, dtype=np.float32)

    # --- 情况 1：1D 输入（计算两个向量之间的相似度） ---
    if A.ndim == 1:
        # 将 A 展平为一维
        A = A.ravel()
        # 计算余弦相似度分母：||v1|| * ||A||
        denom = np.linalg.norm(v1) * np.linalg.norm(A)
        # 如果分母为 0，相似度定义为 0；否则计算点积除以分母
        return float(0.0 if denom == 0 else np.dot(v1, A) / denom)

    # --- 情况 2：2D 输入（计算 v1 与矩阵 A 中每一行的相似度） ---
    # 确保 A 至少是二维的
    A = np.atleast_2d(A)
    # 计算 v1 的范数（长度）
    v1_norm = np.linalg.norm(v1)
    # 沿行方向（axis=1）计算 A 中每个向量的范数
    A_norms = np.linalg.norm(A, axis=1)
    # 计算分母数组
    denom = v1_norm * A_norms
    
    # 开启 NumPy 错误状态管理，忽略除以零或无效值的警告
    with np.errstate(divide='ignore', invalid='ignore'):
        # 使用矩阵乘法 @ 计算 v1 与 A 中各行的点积，并除以分母
        # np.where 确保分母为 0 时返回 1.0 以避免报错，随后会统一处理
        sims = (A @ v1) / np.where(denom == 0, 1.0, denom)
    
    # 将原本分母为 0 的对应位置的相似度分值强制设为 0.0
    sims[denom == 0] = 0.0
    # 将计算结果（NumPy 数组）转换为 Python 列表并返回
    return sims.tolist()


def top_k_greatest_indices(lst, k):
    """
    获取列表中最大的前 k 个元素的索引。

    参数:
    lst (list): 需要评估的元素列表。
    k (int): 需要提取的前几名元素的数量。

    返回:
    list: 对应 lst 中数值最大的前 k 个元素的索引列表。
    """
    # 使用 enumerate 将列表转换为 (索引, 值) 的元组对，以便在排序后追踪原始位置
    indexed_list = list(enumerate(lst))
    # 根据值（x[1]）对元组列表进行降序排列（reverse=True）
    sorted_by_value = sorted(indexed_list, key=lambda x: x[1], reverse=True)
    # 从排序后的列表中提取前 k 个元组，并仅获取它们的索引
    top_k_indices = [index for index, value in sorted_by_value[:k]]
    # 返回最终的索引列表
    return top_k_indices

Now let's define the retriever function.

现在让我们定义检索器函数。

In [9]:
def retrieve_documents(query, embeddings, model, top_k=5):
    """
    使用余弦相似度检索与查询语句最相似的前 k 个文档。
    
    假设条件：
      - 已在外部定义了 preprocess_text, top_k_greatest_indices, df 和 newsgroups_train。
      - embeddings 是一个包含文档向量的可迭代对象（NumPy 数组或 PyTorch 张量）。
      - model.encode 函数支持 convert_to_tensor 参数（例如 sentence-transformers 库）。
    """
    
    # 1. 预处理：清洗用户输入的查询词（如去除首尾空格）
    query_clean = preprocess_text(query)
    
    # 2. 向量化：将清洗后的文本转换为 32 位浮点数的 NumPy 向量
    # convert_to_tensor=False 确保直接获得 NumPy 数组而不是 PyTorch 张量
    query_embedding = model.encode(query_clean, convert_to_tensor=False).astype(np.float32)

    # 初始化一个列表，用于存储每个文档与查询词的相似度分数
    cosine_scores = []
    
    # 3. 循环遍历：逐个计算查询向量与数据库中每个文档向量的相似度
    for x in embeddings:
        # 确保每个文档向量都是 NumPy 数组格式
        if hasattr(x, "detach"):  # 如果是 PyTorch 张量
            x = x.detach().cpu().numpy() # 剥离计算图并移至 CPU 转为 NumPy
        
        # 统一将向量转换为 float32 类型的 NumPy 数组
        x = np.asarray(x, dtype=np.float32)

        # 调用之前定义的 cosine_similarity 函数计算相似度得分
        # 这里 x 是一维向量，所以返回一个浮点数分数
        score = cosine_similarity(query_embedding, x) 
        
        # 将得分转化为浮点数并存入列表
        cosine_scores.append(float(score))

    # 4. 排序：利用之前定义的函数获取相似度分数最高的前 k 个索引
    top_results = top_k_greatest_indices(cosine_scores, k=top_k)

    # 5. 输出：展示搜索结果
    print(f"查询语句: {query}")
    print("-" * 30)
    for idx in top_results:
        # 根据索引从原始 DataFrame (df) 中提取匹配的文本内容（展示前 200 字符）
        print(f"文档内容: {df.iloc[idx]['text'][:200]}...")
        # 提取并展示该文档所属的分类名称
        print(f"所属类别: {newsgroups_train.target_names[df.iloc[idx]['category']]}...")
        # 换行美化输出结果
        print("\n\n")

        
# --- 示例用法 ---
# 定义一个测试查询词
example_query = "space exploration"
# 执行检索函数，传入之前存好的向量库和模型，获取得分最高的前 2 个结果
retrieve_documents(example_query, embedding_vectors, model, top_k = 2)

查询语句: space exploration
------------------------------
文档内容: From: u1452@penelope.sdsc.edu (Jeff Bytof - SIO)
Subject: End of the Space Age?
Organization: San Diego Supercomputer Center @ UCSD
Lines: 16
Distribution: world
NNTP-Posting-Host: penelope.sdsc.edu

...
所属类别: sci.space...



文档内容: From: dennisn@ecs.comm.mot.com (Dennis Newkirk)
Subject: Space class for teachers near Chicago
Organization: Motorola
Distribution: usa
Nntp-Posting-Host: 145.1.146.43
Lines: 59

I am posting this for...
所属类别: sci.space...





<a id='2'></a>
## 2 - Retrieving metrics
## 2 -检索指标
---

Let's explore briefly the most common metrics for retrieval systems: Precision@K and Recall@K.

让我们简要探讨检索系统的最常见指标：Precision@K和Recall@K。

<a id='2-1'></a>
### 2.1 Precision@K
### 2.1 Precision@K

Precision@K provides an evaluation of the relevancy of the top K retrieved documents. It's calculated as the ratio of relevant documents in the top K results to K (the total number of documents retrieved).

Precision@K提供了对检索到的前K个文档的相关性的评估。它被计算为前K个结果中相关文档与K（检索的文档总数）的比率。

$$\text{Precision@K} = \frac{\text{Number of Relevant Documents in Top K}}{\text{K}}$$

where K is the number of documents retrieved.

其中K是检索到的文档数。

In [10]:
def precision_at_k(relevant_count, k):
    """
    为检索系统计算 Precision@K（前 K 项精确率）。

    Precision@K 是指在前 K 个检索出的文档中真正相关文档的数量，
    占这前 K 个被检索出来的文档总数（也就是 K）的比例。

    参数:
        relevant_count (int): 前 K 个结果中相关文档的数量。
        k (int): 系统检索出的文档总数（即前 K 个）。

    返回:
        float: Precision@K 的值。如果 k 为 0，则返回 0.0。
    
    异常:
        ValueError: 如果输入值中有负数，或者 relevant_count 大于 k，则抛出异常。
    """
    # 检查输入合法性：确保相关文档数量和 K 都不是负数
    if relevant_count < 0 or k < 0:
        raise ValueError("所有输入值必须为非负数。")
        
    # 逻辑检查：前 K 个结果里的相关文档数，绝对不可能大于 K 本身
    if relevant_count > k:
        raise ValueError("相关文档数量 (relevant_count) 不能大于检索出的文档总数 (k)。")

    # 边界情况处理：如果系统没有检索返回任何结果（k 为 0），
    # 为了避免除以零错误，直接返回 0.0
    if k == 0:
        return 0.0

    # 核心公式：前 K 个中找回的相关数 / 检索出的总数 K
    # 例如：你在搜索引擎搜东西，只看第一页的前 5 条结果（K=5），
    # 发现其中有 3 条是你真正想要的内容，另外 2 条是废话。
    # 那么 Precision@5 = 3 / 5 = 0.6 (60%)
    return relevant_count / k

<a id='2-2'></a>
### 2.2 Recall@K
### 2.2 Recall@K

Recall@K evaluates the retrieval system's ability to find all relevant documents from the dataset within the top K results. It's calculated as the ratio of relevant documents in the top K results to the total number of relevant documents in the entire corpus.

Recall@K评估检索系统从前K个结果的数据集中查找所有相关文档的能力。它是计算前K个结果中相关文档与整个语料库中相关文档总数的比率。

$$\text{Recall@K} = \frac{\text{Number of Relevant Documents in Top K}}{\text{Total Number of Relevant Documents in Corpus}}$$

In [11]:
def recall_at_k(relevant_count, total_relevant):
    """
    为检索系统计算 Recall@K（前 K 项召回率）。

    Recall@K 是指在前 K 个检索出的文档中相关文档的数量，
    与整个语料库（数据库）中相关文档总数的比例。

    参数:
        relevant_count (int): 前 K 个结果中相关文档的数量。
        total_relevant (int): 整个语料库中相关文档的总数。

    返回:
        float: Recall@K 的值。如果 total_relevant 为 0，则返回 0.0。
    
    异常:
        ValueError: 如果输入值中有负数，则抛出异常。
    """
    # 检查输入合法性：确保相关文档数量和总数都不是负数
    if relevant_count < 0 or total_relevant < 0:
        raise ValueError("所有输入值必须为非负数。")

    # 边界情况处理：如果整个数据库里根本没有相关文档（总数为 0），
    # 为了避免除以零错误，直接返回 0.0
    if total_relevant == 0:
        return 0.0

    # 核心公式：前 K 个中找回的相关数 / 数据库中所有的相关数
    # 例如：数据库里共有 10 篇关于“空间”的文章，你在前 5 个结果里找到了 2 篇，
    # 那么 Recall@5 = 2 / 10 = 0.2 (20%)
    return relevant_count / total_relevant

<a id='2-3'></a>
### 2.3 Computing metrics over some queries

Now let's compute these metrics on some pre-defined queries.

### 2.3计算某些查询的度量
现在让我们在一些预定义的查询上计算这些指标。

In [12]:
# 定义更复杂的测试查询语句及其对应的理想类别（标准答案）
test_queries = [
    # query: 用户输入的搜索词 | desired_category: 我们期望系统找回的正确类别标签
    
    {"query": "advancements in space exploration technology", "desired_category": "sci.space"},
    # 意为：太空探索技术的进展 -> 对应类别：科学.太空
    
    {"query": "real-time rendering techniques in computer graphics", "desired_category": "comp.graphics"},
    # 意为：计算机图形学中的实时渲染技术 -> 对应类别：计算机.图形
    
    {"query": "latest findings in cardiovascular medical research", "desired_category": "sci.med"},
    # 意为：心血管医学研究的最新发现 -> 对应类别：科学.医学
    
    {"query": "NHL playoffs and team performance statistics", "desired_category": "rec.sport.hockey"},
    # 意为：NHL 季后赛与球队表现统计 -> 对应类别：娱乐.运动.冰球
    
    {"query": "impacts of cryptography in online security", "desired_category": "sci.crypt"},
    # 意为：密码学对网络安全的影响 -> 对应类别：科学.加密
    
    {"query": "the role of electronics in modern computing devices", "desired_category": "sci.electronics"},
    # 意为：电子技术在现代计算设备中的作用 -> 对应类别：科学.电子
    
    {"query": "motorcycles maintenance tips for enthusiasts", "desired_category": "rec.motorcycles"},
    # 意为：给爱好者的摩托车保养建议 -> 对应类别：娱乐.摩托车
    
    {"query": "high-performance baseball tactics for championships", "desired_category": "rec.sport.baseball"},
    # 意为：夺冠所需的高性能棒球战术 -> 对应类别：娱乐.运动.棒球
    
    {"query": "historical influence of politics on society", "desired_category": "talk.politics.misc"},
    # 意为：政治对社会的历史影响 -> 对应类别：谈话.政治.综合
    
    {"query": "latest technology trends in the Windows operating system", "desired_category": "comp.os.ms-windows.misc"}
    # 意为：Windows 编译系统的最新技术趋势 -> 对应类别：计算机.操作系统.Windows
]

In [13]:
def compute_metrics(queries, embeddings, model, top_k=5):
    """
    针对查询列表，计算文档向量集的 Precision@K 和 Recall@K。
    
    假设条件：
      - preprocess_text, top_k_greatest_indices, precision_at_k, recall_at_k, df, newsgroups_train 已在外部定义。
      - embeddings 是文档向量的可迭代对象（NumPy 数组或 PyTorch 张量）。
      - model.encode 支持 convert_to_tensor 参数。
    """

    results = [] # 用于存储每个查询的评测结果

    # --- 性能优化：将所有零散向量一次性标准化为 NumPy 矩阵 ---
    np_embeddings = []
    for x in embeddings:
        if hasattr(x, "detach"):  # 如果是 PyTorch 张量，先转为 NumPy
            x = x.detach().cpu().numpy()
        # 确保向量是一维的，并存入列表
        np_embeddings.append(np.asarray(x, dtype=np.float32).ravel())
    
    # np.vstack 作用：将列表中的 N 个向量“垂直堆叠”，形成一个 (N, D) 的二维矩阵
    # 这样后续可以用矩阵乘法一次性算出所有相似度，速度比循环快得多
    E = np.vstack(np_embeddings)  

    # --- 开始遍历每个测试查询 ---
    for item in queries:
        query = item["query"] # 提取搜索词
        desired_category = item["desired_category"] # 提取我们期望的正确类别

        # 1. 预处理并编码查询词
        q_clean = preprocess_text(query)
        # 强制 convert_to_tensor=False 获得 NumPy 数组，避免后续转换错误
        q_emb = model.encode(q_clean, convert_to_tensor=False)
        q_emb = np.asarray(q_emb, dtype=np.float32).ravel()

        # 2. 计算向量相似度：这里调用的是矩阵运算版本的 cosine_similarity
        # 会返回一个长度为 N 的列表，包含查询词与所有文档的相似度分数
        cosine_scores = cosine_similarity(q_emb, E)

        # 3. 获取前 K 个最相似文档的索引
        top_results = top_k_greatest_indices(cosine_scores, k=top_k)

        # 4. 获取这 K 个结果对应的实际类别名称
        retrieved_categories = [
            newsgroups_train.target_names[df.iloc[idx]["category"]] for idx in top_results
        ]

        # 5. 计算核心指标
        # 计算前 K 个结果中有多少个类别与“期望类别”一致（即相关文档数）
        relevant_in_top_k = sum(1 for cat in retrieved_categories if cat == desired_category)
        
        # 计算整个语料库中属于该“期望类别”的文档总数（用于计算召回率 Recall）
        total_relevant_in_corpus = sum(
            1 for idx in range(len(df))
            if newsgroups_train.target_names[df.iloc[idx]["category"]] == desired_category
        )

        # 6. 调用之前定义的函数计算 P 和 R
        p = precision_at_k(relevant_in_top_k, top_k)
        r = recall_at_k(relevant_in_top_k, total_relevant_in_corpus)

        # 将当前查询的得分存入结果列表
        results.append({
            "query": query,
            "precision@k": p,
            "recall@k": r,
        })

    return results

In [ ]:
# 定义不同的 K 值，用于对比实验
k_values = [5, 20, 50]

# 遍历每一个 K 值（分别测试返回 5 条、20 条和 50 条结果时的情况）
for k in k_values:
    # 打印分割线和当前的 K 值，使输出结果清晰易读
    print(f"\n{'='*80}")
    print(f"展示 K={k} 时的检索结果：")
    print('='*80)
    
    # 调用之前定义的 compute_metrics 函数，计算当前 K 值下的精确率和召回率
    # 传入测试查询集、预计算的向量库、模型以及当前的 K
    results = compute_metrics(test_queries, embedding_vectors, model, top_k=k)
    
    # 遍历并显示每一个测试查询的具体得分
    for result in results:
        # 打印查询词
        print(f"查询语句: {result['query']}")
        # 打印对应的 Precision@K 和 Recall@K（保留两位小数）
        print(f"  精确率 Precision@{k}: {result['precision@k']:.2f}, 召回率 Recall@{k}: {result['recall@k']:.2f}")
        # 打印一个空行，方便阅读下一个查询结果
        print()


展示 K=50 时的检索结果：
查询语句: advancements in space exploration technology
  精确率 Precision@50: 1.00, 召回率 Recall@50: 0.08

查询语句: real-time rendering techniques in computer graphics
  精确率 Precision@50: 0.88, 召回率 Recall@50: 0.08

查询语句: latest findings in cardiovascular medical research
  精确率 Precision@50: 1.00, 召回率 Recall@50: 0.08

查询语句: NHL playoffs and team performance statistics
  精确率 Precision@50: 0.98, 召回率 Recall@50: 0.08

查询语句: impacts of cryptography in online security
  精确率 Precision@50: 1.00, 召回率 Recall@50: 0.08

查询语句: the role of electronics in modern computing devices
  精确率 Precision@50: 0.66, 召回率 Recall@50: 0.06

查询语句: motorcycles maintenance tips for enthusiasts
  精确率 Precision@50: 0.98, 召回率 Recall@50: 0.08

查询语句: high-performance baseball tactics for championships
  精确率 Precision@50: 1.00, 召回率 Recall@50: 0.08

查询语句: historical influence of politics on society
  精确率 Precision@50: 0.52, 召回率 Recall@50: 0.06

查询语句: latest technology trends in the Windows operating system
  精确率 Precisio

# 检索结果深入解读：精确率与召回率的权衡

这份评估结果清晰地展示了在检索系统中，随着 **K 值**（返回结果数量）从 5 增加到 20 再到 50 时，**精确率（Precision）与召回率（Recall）之间的权衡关系**：

---

### 1. Precision@K 趋势（通常随 K 的增加而下降）

精确率衡量的是“给出的结果中有多少是相关的”。

* **在 K=5 时**：大多数查询都表现出极高的精确率（0.80-1.00），其中 10 个查询中有 8 个达到了 **满分精确率 (1.00)**。这说明系统排在最前面的结果质量极高。
  
* **在 K=20 时**：部分查询的精确率开始出现下滑：
    * “计算设备中的电子技术”：从 1.00 降至 **0.80**
    * “Windows 操作系统”：从 0.80 降至 **0.65**
    * “摩托车保养”：从 1.00 降至 **0.95**
  
* **在 K=50 时**：随着检索范围扩大，精确率进一步稀释：
    * “计算机图形学”：从 1.00 降至 **0.88**
    * “计算设备中的电子技术”：降至 **0.66**
    * “Windows 操作系统”：降至 **0.60**
    * “政治的历史影响”：维持在 **0.50-0.52** 左右（这是所有测试项中的最低水平）。

---

### 2. Recall@K 趋势（随 K 的增加而上升）

召回率衡量的是“在所有相关的文档中，你找回了多少”。



* **在 K=5 时**：召回率极低（约 0.01 或 **1%**），意味着此时你只触达了数据库中海量相关文档的冰山一角。
  
* **在 K=20 时**：召回率翻了三倍，达到约 **0.03 (3%)**，捕捉到了更多的相关样本。
  
* **在 K=50 时**：召回率提升至 **0.05-0.08 (5-8%)**。虽然绝对值仍不高，但相比 K=5，你找回的相关文档数量增加了约 8 倍。

---

### 核心观察结论

1.  **权衡关系（Tradeoff）显而易见**：
    随着 K 的增加，我们能够找回更多比例的相关文档（**召回率提高**），但代价是不可避免地混入了更多不相关的噪音文档（**精确率降低**）。

2.  **某些查询更具挑战性**：
    关于“政治对社会的历史影响”的查询始终表现出最低的精确率（0.40-0.52）。这表明该查询词在语义上可能比较模糊，或者 `talk.politics.misc` 这一类别的内容与其他类别（如历史、宗教）有较大的重叠，导致模型难以精准区分。

3.  **对 RAG（检索增强生成）系统的启示**：
    对于 RAG 系统而言，**K=5 到 K=20 通常是最佳平衡点**。
    * 这些值能保证极高的精确率，确保喂给大模型（LLM）的信息是准确的。
    * 它能将上下文长度控制在合理范围内。
    * **核心逻辑**：在 RAG 中，我们的目标是找到“最强有力”的证据，而不是找回“所有”相关的废话。

4.  **召回率的天然限制**：
    即便在 K=50 时，召回率依然只有不到 10%。这是因为数据集每个类别有 500-600 篇文档。要显著提升召回率，需要将 K 提升到几百，但这会瞬间破坏精确率，在实际应用（尤其是算力受限的情况下）中是不可取的。

---

**总结**：如果你追求“首屏体验”或“大模型输入质量”，请坚持使用较小的 K 值；如果你是在做“学术查重”或“法律取证”，则需要忍受低精确率并扩大 K 值以换取高召回。

Congratulations on finishing this Ungraded Lab! Keep it up!

祝贺你完成了这个未评分的实验！保持下去！